In [2]:
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader

In [44]:
with open("the-verdict.txt", "r") as file:
    data = file.read()

len(data)

20479

In [4]:
tokenizer = tiktoken.get_encoding("gpt2")

In [5]:
token_ids = tokenizer.encode(data)
len(token_ids)

5145

In [18]:
max_len = 4
stride = 2
for i in range(4):
    ip_token_ids = token_ids[i : max_len + i]
    out_token_ids = token_ids[max_len + i]
    print(
        f"Input: {ip_token_ids} {tokenizer.decode(ip_token_ids)} --> ",
        f"Output: {out_token_ids} {tokenizer.decode([out_token_ids])} ",
    )

Input: [40, 367, 2885, 1464] I HAD always -->  Output: 1807  thought 
Input: [367, 2885, 1464, 1807]  HAD always thought -->  Output: 3619  Jack 
Input: [2885, 1464, 1807, 3619] AD always thought Jack -->  Output: 402  G 
Input: [1464, 1807, 3619, 402]  always thought Jack G -->  Output: 271 is 


In [93]:
# GPTDataset
class GPTDataset(Dataset):

    def __init__(self, txt, tokenizer, max_len, stride):
        self.input_ids = []
        self.target_ids = []
        all_tokens = tokenizer.encode(txt)
        for i in range(0, len(all_tokens) - max_len, stride):
            ips = all_tokens[i : max_len + i]
            outs = all_tokens[i + 1 : (max_len + i) + 1]
            self.input_ids.append(torch.tensor(ips))
            self.target_ids.append(torch.tensor(outs))

    def __getitem__(self, index):
        return self.input_ids[index], self.target_ids[index]

    def __len__(self):
        return len(self.input_ids)
        # return 5

In [94]:
gpt_dataset = GPTDataset(data, tokenizer, 4, 1)
gpt_dl = DataLoader(
    gpt_dataset,
    batch_size=1,
    shuffle=False,
    drop_last=True,
    num_workers=0,
)

In [96]:
# for x in gpt_dl:
#     print(x)